In [ ]:
from IPython.display import HTML, display

display(HTML("""
<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 60%, #0f3460 100%);
            border-radius: 12px; padding: 40px 40px 30px 40px; margin: 10px 0 30px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <div style="color:#a0aec0; font-size:0.8em; letter-spacing:2px; text-transform:uppercase; margin-bottom:12px;">
    LLM Lab — Section 01b
  </div>
  <h1 style="color:white; font-size:2.2em; margin:0 0 10px 0; font-weight:700; line-height:1.2; border:none;">
    Model Datasheet<br>
    <span style="color:#63b3ed;">What We're Actually Running</span>
  </h1>
  <p style="color:#cbd5e0; font-size:1em; margin:16px 0 24px 0; max-width:600px; line-height:1.6;">
    Read the datasheet before you power on the part. Qwen3.5 isn't a pure
    transformer — it's a hybrid attention architecture. Let's crack it open.
  </p>
  <div style="display:flex; gap:16px; flex-wrap:wrap;">
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">📋 Datasheet</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🧠 Architecture</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">📊 KV Cache Math</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">💾 Memory Budget</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">⏱ ~15 min</span>
  </div>
</div>
"""))

In [ ]:
# Setup — connect to the same 3 MLX servers from Section 01
from openai import OpenAI
import time
import json
import urllib.request
import psutil
import re
import html as html_mod
import markdown
import threading
import concurrent.futures
from IPython.display import HTML, display

MODELS = [
    {"label": "122B", "model": "arthurcollet/Qwen3.5-122B-A10B-mlx-nvfp4", "port": 8800, "color": "#2563eb",
     "hf_id": "Qwen/Qwen3.5-122B-A10B"},
    {"label": "35B", "model": "RepublicOfKorokke/Qwen3.5-35B-A3B-mlx-lm-nvfp4", "port": 8801, "color": "#16a34a",
     "hf_id": "Qwen/Qwen3.5-35B-A3B"},
    {"label": "2B", "model": "RepublicOfKorokke/Qwen3.5-2B-mlx-lm-nvfp4", "port": 8802, "color": "#f59e0b",
     "hf_id": "Qwen/Qwen3.5-2B"},
]

clients = {}
for m in MODELS:
    clients[m["label"]] = OpenAI(base_url=f"http://localhost:{m['port']}/v1", api_key="unused")

print("Clients connected.")

In [ ]:
# Helper functions (same pattern as Section 01)

def strip_think(text):
    """Remove <think>...</think> blocks from Qwen3.5 reasoning output."""
    cleaned = re.sub(r'<think>.*?</think>\s*', '', text, flags=re.DOTALL)
    if '<think>' in cleaned:
        cleaned = cleaned[:cleaned.index('<think>')]
    return cleaned.strip()

def _md(text):
    """Convert markdown text to HTML."""
    try:
        return markdown.markdown(text, extensions=["fenced_code", "tables"])
    except Exception:
        return html_mod.escape(text)

def _render_cards(state, models_order):
    """Render 3-column HTML cards from current state."""
    cards = ""
    for m in models_order:
        s = state[m["label"]]
        text = strip_think(s["text"])
        if not text and not s["done"]:
            rendered = "<em>waiting...</em>"
        elif not text and s["done"]:
            rendered = "<em>(empty response)</em>"
        else:
            rendered = _md(text)
        if s["done"]:
            status = f"{s['tps']:.1f} tok/s"
        else:
            elapsed_str = f"{s['elapsed']:.1f}s" if s["elapsed"] > 0 else ""
            status = f"streaming... {s['tokens']} tok {elapsed_str}"
        cards += f"""
        <div style="flex:1; min-width:250px; background:#f9fafb; border:1px solid #d1d5db;
                    border-left:4px solid {s['color']}; border-radius:0 8px 8px 0; padding:12px 18px;">
          <div style="color:{s['color']}; font-weight:bold; font-size:0.85em; margin-bottom:6px;">
            {m['label']} \u00b7 {status}
          </div>
          <div style="color:#1f2937; line-height:1.6; word-wrap:break-word; font-size:0.9em;">
            {rendered}
          </div>
          <div style="color:#9ca3af; font-size:0.75em; margin-top:8px;">
            {s['tokens']} tokens in {s['elapsed']:.1f}s
          </div>
        </div>"""
    return f'<div style="display:flex; gap:16px; flex-wrap:wrap; margin:10px 0;">{cards}</div>'

def compare_models(prompt, system_prompt=None, **kwargs):
    """Fire the same prompt at all 3 models in parallel with streaming."""
    kwargs.setdefault("max_tokens", 1024)
    kwargs.setdefault("extra_body", {"chat_template_kwargs": {"enable_thinking": False}})
    order = {"2B": 0, "35B": 1, "122B": 2}
    models_order = sorted(MODELS, key=lambda m: order.get(m["label"], 99))
    state = {}
    for m in MODELS:
        state[m["label"]] = {"text": "", "tokens": 0, "elapsed": 0, "tps": 0, "done": False, "color": m["color"]}
    handle = display(HTML(_render_cards(state, models_order)), display_id=True)
    def stream_model(m):
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})
        t0 = time.time()
        try:
            response = clients[m["label"]].chat.completions.create(
                model=m["model"], messages=messages, stream=True, **kwargs
            )
            token_count = 0
            for chunk in response:
                if chunk.choices and chunk.choices[0].delta.content:
                    state[m["label"]]["text"] += chunk.choices[0].delta.content
                    token_count += 1
                    elapsed = time.time() - t0
                    state[m["label"]]["tokens"] = token_count
                    state[m["label"]]["elapsed"] = elapsed
                    state[m["label"]]["tps"] = token_count / elapsed if elapsed > 0 else 0
        except Exception as e:
            state[m["label"]]["text"] = f"Error: {e}"
        finally:
            elapsed = time.time() - t0
            state[m["label"]]["elapsed"] = elapsed
            if state[m["label"]]["tokens"] > 0:
                state[m["label"]]["tps"] = state[m["label"]]["tokens"] / elapsed
            state[m["label"]]["done"] = True
    threads = []
    for m in MODELS:
        t = threading.Thread(target=stream_model, args=(m,))
        t.start()
        threads.append(t)
    while not all(state[m["label"]]["done"] for m in MODELS):
        time.sleep(0.3)
        handle.update(HTML(_render_cards(state, models_order)))
    handle.update(HTML(_render_cards(state, models_order)))
    results = []
    for m in models_order:
        s = state[m["label"]]
        results.append({
            "label": m["label"], "text": strip_think(s["text"]),
            "tokens": s["tokens"], "elapsed": s["elapsed"],
            "tps": s["tps"], "color": s["color"]
        })
    return results

print("Helpers loaded.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📋 Step 1: Architecture Overview</h2>
</div>

Every module in this lab uses Qwen3.5 models. Before we go deeper into tokenization, prompting, RAG, or fine-tuning, we need to understand the hardware we're building on. In embedded terms, this is the **datasheet review** before you start writing firmware.

The critical thing: **Qwen3.5 is not a pure transformer.** It's a hybrid attention architecture that mixes two fundamentally different attention mechanisms.

In [ ]:
# Fetch config.json from HuggingFace for all 3 models
configs = {}
for m in MODELS:
    url = f"https://huggingface.co/{m['hf_id']}/resolve/main/config.json"
    print(f"Fetching {m['label']} config from {m['hf_id']}...", flush=True)
    try:
        raw = json.loads(urllib.request.urlopen(url).read())
        cfg = raw.get("text_config", raw)  # params may be nested
        configs[m["label"]] = cfg
        print(f"  \u2713 {cfg.get('num_hidden_layers', '?')} layers")
    except Exception as e:
        print(f"  \u2717 Failed: {e}")
        configs[m["label"]] = {}

# Build architecture cards
cards = ""
for m in MODELS:
    cfg = configs[m["label"]]
    n_layers = cfg.get("num_hidden_layers", "?")
    n_kv_heads = cfg.get("num_key_value_heads", "?")
    n_attn_heads = cfg.get("num_attention_heads", "?")
    head_dim = cfg.get("head_dim", "?")
    hidden_size = cfg.get("hidden_size", "?")
    model_type = cfg.get("model_type", "?")
    
    # MoE info
    num_experts = cfg.get("num_experts", cfg.get("num_local_experts", None))
    experts_per_tok = cfg.get("num_experts_per_tok", cfg.get("num_selected_experts", None))
    
    # DeltaNet info
    delta_cfg = cfg.get("linear_attention_config", {})
    delta_heads = delta_cfg.get("num_heads", "N/A")
    delta_kv_heads = delta_cfg.get("num_kv_heads", "N/A")
    
    # Calculate GQA vs DeltaNet layers (3:1 ratio)
    if isinstance(n_layers, int):
        n_gqa = n_layers // 4
        n_delta = n_layers - n_gqa
        layer_split = f"{n_delta} DeltaNet + {n_gqa} GQA"
    else:
        layer_split = "?"
    
    moe_line = ""
    if num_experts:
        moe_line = f"<tr><td style='padding:4px 10px; color:#6b7280;'>MoE Experts</td><td style='padding:4px 10px; font-weight:bold;'>{num_experts} total, {experts_per_tok} active/tok</td></tr>"
    
    cards += f"""
    <div style="flex:1; min-width:280px; background:#f9fafb; border:1px solid #d1d5db;
                border-left:4px solid {m['color']}; border-radius:0 8px 8px 0; padding:14px 18px;">
      <div style="color:{m['color']}; font-weight:bold; font-size:1em; margin-bottom:10px;">
        {m['label']} \u2014 {model_type}
      </div>
      <table style="font-size:0.85em; border-collapse:collapse; width:100%;">
        <tr><td style="padding:4px 10px; color:#6b7280;">Layers</td><td style="padding:4px 10px; font-weight:bold;">{n_layers} ({layer_split})</td></tr>
        <tr><td style="padding:4px 10px; color:#6b7280;">GQA Heads</td><td style="padding:4px 10px; font-weight:bold;">{n_attn_heads} attn / {n_kv_heads} KV</td></tr>
        <tr><td style="padding:4px 10px; color:#6b7280;">DeltaNet Heads</td><td style="padding:4px 10px; font-weight:bold;">{delta_heads} heads / {delta_kv_heads} KV</td></tr>
        <tr><td style="padding:4px 10px; color:#6b7280;">Head Dim</td><td style="padding:4px 10px; font-weight:bold;">{head_dim}</td></tr>
        <tr><td style="padding:4px 10px; color:#6b7280;">Hidden Size</td><td style="padding:4px 10px; font-weight:bold;">{hidden_size}</td></tr>
        {moe_line}
      </table>
    </div>"""

display(HTML(f'<div style="display:flex; gap:16px; flex-wrap:wrap; margin:10px 0;">{cards}</div>'))
print("\nAll 3 models share the same 3:1 DeltaNet-to-GQA ratio.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🔀 Step 2: The Hybrid Attention Pattern</h2>
</div>

Qwen3.5 alternates two fundamentally different attention mechanisms in a **3:1 repeating block**:

```
N × (3 × (Gated DeltaNet → FFN) → 1 × (GQA Attention → FFN))
```

- **Gated DeltaNet** (75% of layers): Linear attention with a recurrent state. Like an IIR filter — it maintains a running accumulator that gets updated with each token. O(n) per token, fixed memory.
- **GQA Attention** (25% of layers): Standard grouped-query attention with full softmax. Like RAM — random access to any previous token. O(n²) per token, KV cache grows with context.

The DeltaNet layers handle the bulk of the work cheaply. The GQA "anchor" layers every 4th position provide precise long-range lookups that linear attention can't do well.

In [ ]:
# Visualize the hybrid attention pattern for each model

for m in MODELS:
    cfg = configs[m["label"]]
    n_layers = cfg.get("num_hidden_layers", 0)
    if not isinstance(n_layers, int) or n_layers == 0:
        continue
    
    # Build layer type sequence: every 4th layer is GQA (index 3, 7, 11, ...)
    blocks = ""
    for i in range(n_layers):
        is_gqa = (i % 4 == 3)
        bg = "#2563eb" if is_gqa else "#a855f7"  # blue=GQA, purple=DeltaNet
        label = "G" if is_gqa else "D"
        # Only show labels for first 32 layers to keep it readable
        if n_layers <= 64:
            blocks += f'<span style="display:inline-block; width:12px; height:20px; background:{bg}; margin:1px; border-radius:2px; font-size:6px; color:white; text-align:center; line-height:20px;" title="Layer {i}: {\"GQA\" if is_gqa else \"DeltaNet\"}">{label}</span>'
        else:
            blocks += f'<span style="display:inline-block; width:6px; height:20px; background:{bg}; margin:0.5px; border-radius:1px;" title="Layer {i}: {\"GQA\" if is_gqa else \"DeltaNet\"}"></span>'
    
    n_gqa = n_layers // 4
    n_delta = n_layers - n_gqa
    
    display(HTML(f"""
    <div style="margin:12px 0; padding:12px 16px; background:#f9fafb; border:1px solid #d1d5db; border-left:4px solid {m['color']}; border-radius:0 8px 8px 0;">
      <div style="color:{m['color']}; font-weight:bold; margin-bottom:8px;">{m['label']} \u2014 {n_layers} layers</div>
      <div style="line-height:24px; font-family:monospace;">{blocks}</div>
      <div style="margin-top:8px; font-size:0.8em; color:#6b7280;">
        <span style="color:#a855f7;">\u25a0</span> DeltaNet ({n_delta}) &nbsp;&nbsp;
        <span style="color:#2563eb;">\u25a0</span> GQA ({n_gqa}) &nbsp;&nbsp;
        Pattern repeats every 4 layers: D-D-D-G
      </div>
    </div>
    """))

print("\nThe embedded analogy: DeltaNet is like an IIR filter \u2014 a running accumulator")
print("(constant memory, fast, lossy). GQA is like RAM \u2014 full random access")
print("(growing memory, slower, lossless). The architecture uses the cheap")
print("resource for 75% of layers and the expensive resource at sync points.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📊 Step 3: KV Cache Math</h2>
</div>

The KV cache formula from Section 01 needs a correction for hybrid models. **Only the GQA layers need KV cache** — DeltaNet layers use a fixed-size recurrent state.

```
KV cache (bytes) = 2 × n_gqa_layers × n_kv_heads × head_dim × seq_len × dtype_bytes
```

The hybrid design saves 75% of KV cache memory compared to a hypothetical all-GQA model.

In [ ]:
# KV Cache calculator — hybrid vs naive (all-layers) formula

CONTEXT_LENGTHS = [8_192, 32_768, 131_072]  # 8K, 32K, 128K tokens
DTYPE_BYTES = 2  # float16

rows = ""
for m in MODELS:
    cfg = configs[m["label"]]
    n_layers = cfg.get("num_hidden_layers", 0)
    n_kv_heads = cfg.get("num_key_value_heads", 0)
    head_dim = cfg.get("head_dim", 0)
    
    if not all([n_layers, n_kv_heads, head_dim]):
        continue
    
    n_gqa = n_layers // 4  # Only GQA layers need KV cache
    
    for ctx in CONTEXT_LENGTHS:
        # Hybrid formula (actual): only GQA layers
        kv_hybrid = 2 * n_gqa * n_kv_heads * head_dim * ctx * DTYPE_BYTES
        # Naive formula: if ALL layers used GQA
        kv_naive = 2 * n_layers * n_kv_heads * head_dim * ctx * DTYPE_BYTES
        savings = (1 - kv_hybrid / kv_naive) * 100 if kv_naive > 0 else 0
        
        ctx_label = f"{ctx // 1024}K"
        rows += (
            f"<tr>"
            f"<td style='padding:6px 10px; color:{m['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{m['label']}</td>"
            f"<td style='padding:6px 10px; border-bottom:1px solid #e5e7eb;'>{ctx_label}</td>"
            f"<td style='padding:6px 10px; border-bottom:1px solid #e5e7eb;'>{n_gqa} GQA / {n_layers} total</td>"
            f"<td style='padding:6px 10px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{kv_hybrid / 1e9:.2f} GB</td>"
            f"<td style='padding:6px 10px; color:#dc2626; border-bottom:1px solid #e5e7eb;'>{kv_naive / 1e9:.2f} GB</td>"
            f"<td style='padding:6px 10px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{savings:.0f}%</td>"
            f"</tr>"
        )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:600px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 10px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 10px; color:white; text-align:left;">Context</th>
    <th style="padding:8px 10px; color:white; text-align:left;">Layers</th>
    <th style="padding:8px 10px; color:white; text-align:left;">Hybrid KV</th>
    <th style="padding:8px 10px; color:white; text-align:left;">Naive KV</th>
    <th style="padding:8px 10px; color:white; text-align:left;">Saved</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
<div style="color:#6b7280; font-size:0.8em; margin-top:4px;">
  Formula: 2 \u00d7 n_gqa_layers \u00d7 n_kv_heads \u00d7 head_dim \u00d7 seq_len \u00d7 2 bytes (fp16).
  \"Naive\" = if all layers used GQA attention.
</div>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">💾 Step 4: Memory Budget Calculator</h2>
</div>

Given 128 GB unified RAM with 3 models loaded: how much headroom do we have? What's the maximum safe context length before we start swapping?

In [ ]:
# Memory budget: model weights + KV cache + OS overhead

ram = psutil.virtual_memory()
total_gb = ram.total / 1e9
used_gb = (ram.total - ram.available) / 1e9
available_gb = ram.available / 1e9

# Approximate model footprints (NVFP4 quantization)
MODEL_SIZES = {"122B": 65.0, "35B": 20.0, "2B": 1.2}
total_model_gb = sum(MODEL_SIZES.values())

# OS + system overhead (estimate from current usage minus model weights)
os_overhead_gb = max(used_gb - total_model_gb, 4.0)  # at least 4 GB for OS

# Headroom for KV cache
kv_headroom_gb = total_gb - total_model_gb - os_overhead_gb

print(f"System RAM:     {total_gb:.0f} GB")
print(f"Model weights:  {total_model_gb:.1f} GB (122B={MODEL_SIZES['122B']}, 35B={MODEL_SIZES['35B']}, 2B={MODEL_SIZES['2B']})")
print(f"OS overhead:    ~{os_overhead_gb:.1f} GB")
print(f"KV headroom:    ~{kv_headroom_gb:.1f} GB")
print()

# Calculate max safe context length for each model given the headroom
rows = ""
for m in MODELS:
    cfg = configs[m["label"]]
    n_layers = cfg.get("num_hidden_layers", 0)
    n_kv_heads = cfg.get("num_key_value_heads", 0)
    head_dim = cfg.get("head_dim", 0)
    
    if not all([n_layers, n_kv_heads, head_dim]):
        continue
    
    n_gqa = n_layers // 4
    # Bytes per token of KV cache for this model
    bytes_per_token = 2 * n_gqa * n_kv_heads * head_dim * 2  # 2 for K+V, 2 for fp16
    
    # Max tokens that fit in the KV headroom
    max_tokens = int(kv_headroom_gb * 1e9 / bytes_per_token) if bytes_per_token > 0 else 0
    max_tokens_k = max_tokens / 1024
    
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:{m['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{m['label']}</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>{bytes_per_token:,} bytes/tok</td>"
        f"<td style='padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{max_tokens_k:,.0f}K tokens</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">KV Cost/Token</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Max Context (safe)</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
<div style="color:#6b7280; font-size:0.8em; margin-top:4px;">
  \"Safe\" = fits in remaining RAM ({kv_headroom_gb:.1f} GB) without triggering swap.
  In practice, leave extra margin \u2014 swap kills throughput.
</div>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧩 Step 5: MoE — Why 122B Fits</h2>
</div>

Mixture-of-Experts (MoE) is why a 122B parameter model can run on a 128 GB machine. Each MoE layer contains multiple small "expert" sub-networks, but a learned router only activates a subset per token.

**The embedded analogy:** MoE is like a crossbar switch — one bus selected per cycle. All the expert weights live in memory (122B total), but only ~10B are read per forward pass. The memory footprint is the full model, but the compute (and memory bandwidth) per token is proportional to the active parameters.

In [ ]:
# MoE: total params vs active params per token
# Compare memory read per token across all 3 models

model_specs = [
    {"label": "122B", "color": "#2563eb", "total_params": 122e9, "active_params": 10e9,
     "bytes_per_param": 0.5, "disk_gb": 65, "desc": "MoE: 122B total, 10B active"},
    {"label": "35B", "color": "#16a34a", "total_params": 35e9, "active_params": 3e9,
     "bytes_per_param": 0.5, "disk_gb": 20, "desc": "MoE: 35B total, 3B active"},
    {"label": "2B", "color": "#f59e0b", "total_params": 2e9, "active_params": 2e9,
     "bytes_per_param": 0.5, "disk_gb": 1.2, "desc": "Dense: all 2B active"},
]

# Memory bandwidth of M4 Max
BW_GBS = 546  # GB/s

rows = ""
for spec in model_specs:
    active_gb = spec["active_params"] * spec["bytes_per_param"] / 1e9
    # Theoretical max tok/s = bandwidth / active weight reads per token
    theoretical_tps = BW_GBS / active_gb if active_gb > 0 else 0
    
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:{spec['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{spec['label']}</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>{spec['disk_gb']} GB</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>{active_gb:.1f} GB</td>"
        f"<td style='padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{theoretical_tps:.0f} tok/s</td>"
        f"<td style='padding:6px 12px; color:#6b7280; font-size:0.85em; border-bottom:1px solid #e5e7eb;'>{spec['desc']}</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:600px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">On Disk</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Active/Token</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Theoretical tok/s</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Architecture</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
<div style="color:#6b7280; font-size:0.8em; margin-top:4px;">
  Theoretical = M4 Max bandwidth ({BW_GBS} GB/s) \u00f7 active weight bytes per token.
  Real-world throughput is lower due to attention compute, KV cache reads, and scheduling overhead.
</div>
"""))

print("\nKey insight: 122B is on disk but only reads ~5 GB of active weights per token.")
print("That's why it achieves usable throughput despite being 65 GB on disk.")

In [ ]:
# Ask the models to explain their own architecture
results = compare_models(
    "You are a Qwen3.5 model running on Apple Silicon via MLX. In 2-3 sentences, explain your own architecture "
    "(DeltaNet vs GQA layers, MoE routing if applicable) to an embedded engineer. Use hardware analogies.",
    max_tokens=512,
)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📝 Takeaways</h2>
</div>

- **Qwen3.5 is a hybrid** — not a pure transformer. 75% DeltaNet (linear, fixed memory) + 25% GQA (full attention, growing KV cache)
- **DeltaNet is like an IIR filter** — a running accumulator. Fast, cheap, constant memory. Good for broad patterns, lossy for precise retrieval
- **GQA anchor layers are like RAM** — full random access to all previous tokens. Expensive but precise. Placed every 4th layer as sync points
- **KV cache only grows for GQA layers** — 75% savings vs a hypothetical all-attention model of the same depth
- **MoE is a crossbar switch** — 122B params on disk, ~10B active per token. The memory footprint is the full model, but bandwidth per token is proportional to active params
- **Architecture determines everything** — memory, speed, quality, max context length. Read the datasheet before you build on it

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 What's Next</h2>
</div>

- **Section 01c:** Inference Optimization — speculative decoding, prefix caching, quantization format deep dive, continuous batching
- **Section 02:** Structured output — making LLMs return JSON you can actually parse